# Descriptive Analysis

Goal:
Measure business performance and customer behavior using key metrics.

Focus:
- Revenue trends
- Customer purchasing behavior
- Transaction characteristics

## Load Cleaned Dataset

What:
Load cleaned transaction data from Step 1.

Why:
Ensures all analysis is performed on validated and consistent data.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/ecommerce_cleaned.csv")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SalesAmount,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12


## Create Time Features

What:
Extract YearMonth from InvoiceDate.

Why:
Enables time-based analysis such as monthly trends.

In [2]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")

## Monthly Performance

What:
Calculate total revenue and transactions per month.

Why:
Identifies trends, growth patterns, and seasonality.

In [3]:
monthly_metrics = df.groupby("YearMonth").agg(
    total_sales=("SalesAmount", "sum"),
    total_transactions=("InvoiceNo", "nunique"),
)

monthly_metrics 

,total_sales,total_transactions
YearMonth,,
2010-12,572713.890,1400
2011-01,569445.040,987
2011-02,447137.350,997
2011-03,595500.760,1321
2011-04,469200.361,1149
2011-05,678594.560,1555
2011-06,661213.690,1393
2011-07,600091.011,1331
2011-08,645343.900,1280


## Average Order Value (AOV)

What:
Average revenue per transaction.

Why:
Measures how much customers spend per purchase.

In [4]:
order_values = df.groupby("InvoiceNo")["SalesAmount"].sum()
aov = order_values.mean()

aov

np.float64(480.8659563997409)

## Transactions per Customer

What:
Average number of transactions per customer.

Why:
Indicates how often customers return to purchase.

In [5]:
customer_transactions = df.groupby("CustomerID")["InvoiceNo"].nunique()
transactions_per_customer = customer_transactions.mean()

transactions_per_customer

np.float64(4.272014753342554)

## Items per Transaction

What:
Average number of items per order.

Why:
Shows basket size and purchasing behavior.

Note:
The mean can be distorted by large bulk orders, so we use the median for a more realistic estimate.

In [6]:
items_per_invoice = df.groupby("InvoiceNo")["Quantity"].sum()

items_per_transaction = items_per_invoice.median()

items_per_transaction

np.float64(155.0)

In [7]:
# Outliers

items_per_invoice.sort_values(ascending=False).head(10)

InvoiceNo
581483    80995
541431    74215
556917    15049
563076    14730
572035    13392
567423    12572
552883    12266
563614    12196
562439    11848
548011    11116
Name: Quantity, dtype: int64

### Insight: Basket Size

- The median number of items per transaction is approximately 155.
- This indicates that customers frequently place bulk orders rather than small purchases.
- The presence of extreme outliers (e.g., orders with tens of thousands of items) further suggests this dataset represents wholesale or high-volume purchasing behavior.

## Repeat Customer Rate

What:
Percentage of customers who make more than one purchase.

Why:
Measures customer retention and loyalty.

In [12]:
repeat_customers = (customer_transactions > 1).sum()
total_customers = customer_transactions.count()

repeat_rate = repeat_customers / total_customers

repeat_rate

np.float64(0.6558321807284463)

## Top Countries

What:
Revenue by country.

Why:
Identifies key geographic markets.

In [9]:
top_countries = df.groupby("Country")["SalesAmount"].sum().sort_values(ascending=False).head(10)

top_countries

Country
United Kingdom    7308391.554
Netherlands        285446.340
EIRE               265545.900
Germany            228867.140
France             209024.050
Australia          138521.310
Spain               61577.110
Switzerland         56443.950
Belgium             41196.340
Sweden              38378.330
Name: SalesAmount, dtype: float64

## Export Data for Excel Dashboard

What:
Export KPIs and summary tables.

Why:
Used to build business dashboards in Excel.

In [ ]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Transactions",
        "Total Customers",
        "Average Order Value",
        "Repeat Customer Rate",
    ],
    "Value": [
        df["SalesAmount"].sum(),
        df["InvoiceNo"].nunique(),
        df["CustomerID"].nunique(),
        aov,
        repeat_rate,
    ]
})

kpi_summary.to_excel("../excel/KPI_Summary.xlsx", index=False)

monthly_metrics.to_excel("../excel/Monthly_Metrics.xlsx")
top_countries.to_excel("../excel/Top_Countries.xlsx")